# The evidence model

A finding states a property of the data. Its evidence records identify the
detectors that found it and retain their measurements. When two detectors
find the same statement, the report combines their evidence in one finding.

This chapter explains how statements are identified, how evidence is
structured, how equivalent statements are consolidated, and how findings
are selected for the report.


## One statement, several evidence records

The example plants a conditional relationship: premium segments almost
always ship express. PRISM and the association miner discover it, and the
shared rule evaluator measures it on a common partition:


In [ ]:
import json
import random

import polars as pl

import onceover

rng = random.Random(11)
n = 4000
segment = ["premium" if rng.random() < 0.5 else "std" for _ in range(n)]
shipping = [
    "express" if (rng.random() < 0.95 if s == "premium" else rng.random() < 0.05)
    else "ground"
    for s in segment
]
df = pl.DataFrame({"segment": segment, "shipping": shipping})

report = onceover.profile(df)
rule = next(f for f in report.findings if f["category"] == "conditional_rules")
print("claim:", rule["claim"])
print("conditions:", json.dumps(rule["specification"]["conditions"]))
print("outcome:", json.dumps(rule["specification"]["outcome"]))
print("detectors contributing evidence:", [e["method"]["detector"] for e in rule["evidence"]])

Each evidence record retains its measurements: PRISM's score on uncovered
positives, the association miner's train/holdout counts, and the
evaluator's partition counts. Counts are kept separate. The displayed
claim uses the common evaluator's measurements: eligible rows,
antecedent matches, joint matches, conditional rate, baseline rate, and lift.


## The three dimensions of evidence

Every evidence record describes its basis, scope, and validation separately.

| Dimension | Values | What it answers |
|---|---|---|
| `basis` | `measured`, `estimated`, `inferred` | What kind of support is this — a count or comparison, a fitted quantity validated on held-out data, or an interpretation suggested by measurements? |
| `scope` | `sampling` (`full`/`sample`), `population_rows`, `rows_evaluated` | Which rows were actually covered, out of how many? |
| `validation` | `direct_comparison`, `cross_validation`, `diagnostic_holdout`, `diagnostic_partition`, `none` | How was the statement checked, and was the check independent? |

A `measured` basis with `full` scope and `direct_comparison` records a
count or comparison over all eligible rows, such as key uniqueness or
identity residuals. An `estimated` finding carries the sample and holdout sizes so a
reader can judge it. An `inferred` role ("this column behaves like an
identifier") stays an interpretation even when its supporting counts cover
the full dataset: basis and scope are independent claims.

The `validation` field also records `used_for_selection`. A diagnostic
holdout used to choose a reported model also participated in selection; it
is not an independent confirmation sample.

## Consolidation

Two findings consolidate when they state the same thing. Equivalence is
decided by a canonical key: the category, the type, the subject (columns,
inputs, target), and the specification (form, conditions, outcome,
definitions) — after normalization. Rank, detector, and interestingness are
deliberately absent from the key, so a statement keeps its identity no
matter who discovered it or where it ranks.

When two evidence records arrive for one statement, the report keeps the
claim from the candidate with the broader evaluated coverage, appends all
evidence records, and fills in a missing interpretation if one arrives.
Counts from overlapping samples or different methods remain separate,
each labeled with its own scope.

Distinct statements stay distinct. `X → Y` and `Y → X` are different
claims. `X → Y` and `(X, Z) → Y` are different claims. "If X = a then Y = b"
differs from "X determines Y": the conditional statement applies only
where X = a, while the dependency describes a mapping across all values
checked.

## Selection

The report selects before it renders. Selection ranks every distinct
statement by a deterministic score with three ingredients:

- **evidence quality** — direct full-scope comparison first, then
  cross-validated or holdout-validated estimates, then bare measured or
  inferred support;
- **effect strength**, in units native to each category — zero-violation
  dependencies score highest, rules are scored by improvement over
  baseline weighted by support, predictive dependencies by holdout R² or
  accuracy gain, associations by the strongest measured coefficient,
  concentration by the dominant value's share;
- **simplicity and specificity** — a stated form (a value mapping, an
  affine formula, a piecewise model) outranks an unspecified one; more
  inputs or conditions discount slightly.

Report-wide limits apply across detectors: at most 100 findings, 20
conditional rules, 2 rules per outcome, and 3 predictive findings per
target. The `selection` block records how many candidates entered, how
many survived verification, how many distinct statements they formed, and
how many were selected — so a reader can tell consolidation from
suppression.

In [ ]:
print(json.dumps(report.data["selection"], indent=2))
from collections import Counter
print("selected by category:", dict(Counter(f["category"] for f in report.findings)))

## Limits of the evidence

The report provides no p-values or multiple-testing guarantees. A
conditional rate on a diagnostic partition describes those rows, which
may also have been used during discovery. A missing finding can mean that
a search found no qualifying pattern, reached a cap, or never ran.
Coverage records which passes ran and their search limits
([](../usage/coverage.ipynb)); it cannot establish why an undetected
pattern was missed.

The field-level reference for these structures is the
[report contract](../appendix/report_contract.ipynb). See also the schema
reference in
[docs/findings.md](https://github.com/fcela/onceover/blob/main/docs/findings.md).
